# Excited and Ground States for H2+ Using FCI

In [ ]:
#makes a file containing all the energy for each bond length

# h2plus_fci_pec.py
import psi4
import numpy as np
import csv

#choose basis (small to large):
#cc-pVDZ, cc-pVTZ, cc-pVQZ, cc-pV5Z,
#aug-cc-pVDZ, aug-cc-pVTZ, aug-cc-pVQZ, aug-cc-pV5Z"

basis = "aug-cc-pV5Z"

roots = 11                                      #number of potential energy curves to calculate, including ground state

r_min, r_max, n_points = 0.2, 5.0, 500          #max and min bond length, number of points
csv_out = "h2plus_fci_pec.csv"                   #name csv output file
psi4_out = "h2plus_fci_scan.out"                 #name psi4 output file

psi4.set_output_file(psi4_out, False)            #(Names output file, overrides existing files)
psi4.set_memory("4 GB")                          #How much ram psi4 is allowed to use
psi4.set_num_threads(4)                          #number of threads to use


#create an array of points
bond_lengths = np.linspace(r_min, r_max, n_points)

#run scan for each root at a set R
def run_h2plus_fci(R, roots, basis):        
    #create molecule
    mol = psi4.geometry(f"""
        1 2                               #charge, multiplicity
        H                                 #first hydrogen at (0,0,0)
        H 1 {R}                           #second hydrogen at distance R from hydrogen #1
        units angstrom                    #units of R in angstroms
        symmetry c1                       #turns off symmetry
        """)

    #set up psi4
    psi4.core.clean_options()        #resets options
    psi4.set_options({
        "basis": basis,              #set basis
        "reference": "rohf",         #Restricted Open-Shell Hartree-Fock, needed for open shell system, can also use uhf
        "guess": "sad",              #initial guess
        "scf_type": "pk",            #compute exact integrals
        "maxiter": 200,              #max iterations
        "num_roots": roots,
    })
    psi4.core.clean()                     #removes scratch files, clears data
    psi4.energy("fci", molecule=mol)      #calculate energy using Full Configuration Interaction for my mol
    vars_dict = psi4.core.variables()     #creates a dictionary with psi4 variables

    energies = []                         #create an empty list for energies

    #add each calculated root to the energies list
    for n in range(roots):
        energies.append(vars_dict[f"CI ROOT {n} TOTAL ENERGY"])

    #returns a list of the energies of each root at the n values
    return energies

#empty list for rows
rows = []

#run psi4 for each R
for R in bond_lengths:
    try:
        energies = run_h2plus_fci(R, roots, basis)   #runs function for R
        rows.append([R] + energies)                  #adds ([R], [energies]) to my list of rows
        #print(f"R = {R:.3f} done")                  #print when done
    except Exception as err:                         #If it fails
        print(f"R = {R:.3f} failed: {err}")          #print if it fails
        rows.append([R] + [np.nan] * roots)          #leave a gap for failure
    finally:
        psi4.core.clean()
        psi4.core.clean_options()


headers = ["R_angstrom"]                             #list for headers, starting with R
headers += [f"E_root{i}" for i in range(roots)]       #Add a new header for each root

#write csv file
with open(csv_out, "w", newline="") as f:            #opens csv file, overwrite existing file, prevent blank lines
    w = csv.writer(f)                                #make csv writer
    w.writerow(headers)                              #make row with header
    w.writerows(rows)                                #add data to csv file

print("Done. Wrote h2plus_fci_pec.csv")  

R = 0.200 failed: 
Fatal Error: PSIO_ERROR: 18 (Incorrect block end address)

Practical advice: This error is probably disk related. First, check that your scratch isn’t full.
Second, make sure that you’re setting scratch ("psi4 -s" or "export PSI_SCRATCH") rather than letting it 
default to /tmp. Third, run the job again to see if the error is reproducible.
If it persists, finally, file an issue.

Error occurred in file: /Users/runner/miniforge3/conda-bld/psi4_1767989010891/work/psi4/src/psi4/libpsio/error.cc on line: 142
The most recent 5 function calls were:




PSIO_ERROR: Attempt to write into next entry: 351, buffer_ 0
PSIO_ERROR: unit = 351, errval = 18
PSIO_ERROR: 18 (Incorrect block end address)

Practical advice: This error is probably disk related. First, check that your scratch isn’t full.
Second, make sure that you’re setting scratch ("psi4 -s" or "export PSI_SCRATCH") rather than letting it 
default to /tmp. Third, run the job again to see if the error is reproducible.
If it persists, finally, file an issue.

